# Findings Report — Olist E-Commerce Dataset

**Data scope:** October 2016 – August 2018, 96,478 delivered orders, 15.42M BRL in payment volume.

**Methodology notes:**
- Unless stated otherwise, all analyses are filtered on `order_status = 'delivered'`.
- Order counts use `COUNT(DISTINCT order_id)`. Because an order can have more than one payment row when joined to `payments`, a flat `COUNT()` inflates the order count by ~3-4% (e.g. November 2017: flat count 7,593, actual 7,289).
- 96,470 of the 96,478 delivered orders have a delivery date; delivery time and promise-adherence calculations are based on these 96,470 orders.
- Revenue is measured in two ways: **payment volume** (`payments.payment_value`, including freight and installment charges) and **product revenue** (`order_items.price`, excluding freight). Each finding states which one is used.
- The dataset ends in August 2018. September 2018 has only 16 orders and October 2018 only 4 — this is a data cutoff, not a decline. The last month of the series is not used in any trend interpretation.
- The definitions of two metrics were revised after errors in the first measurement were corrected: **late delivery** is compared at day level (`CAST(... AS DATE)`), and a **repeat purchase** is counted by purchase day rather than by order. The rationale is in the definition notes under Question 4 and Question 5.

**Charts:** The charts in this report are generated in `03_eda_charts.ipynb` and saved to the `reports/` folder; this notebook only embeds them. The numerical basis for each finding is the query output here, independent of the chart — if the chart and the text disagree, the query output prevails.

In [1]:
import duckdb

conn = duckdb.connect()

for table, file in [
    ('orders', 'olist_orders_dataset'),
    ('customers', 'olist_customers_dataset'),
    ('order_items', 'olist_order_items_dataset'),
    ('payments', 'olist_order_payments_dataset'),
    ('reviews', 'olist_order_reviews_dataset'),
    ('products', 'olist_products_dataset'),
    ('sellers', 'olist_sellers_dataset'),
]:
    conn.execute(f"CREATE TABLE {table} AS SELECT * FROM read_csv_auto('../data/raw/{file}.csv')")

print('Tables loaded.')

Tables loaded.


---

## Question 1: How did orders and revenue develop over time?

In [2]:
# Year-over-year comparison: 2017 vs 2018 values for the same calendar month
yoy = conn.execute("""
    SELECT
        strftime(o.order_purchase_timestamp, '%m') AS month,
        COUNT(DISTINCT CASE WHEN YEAR(o.order_purchase_timestamp) = 2017 THEN o.order_id END) AS orders_2017,
        COUNT(DISTINCT CASE WHEN YEAR(o.order_purchase_timestamp) = 2018 THEN o.order_id END) AS orders_2018
    FROM orders o
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY 1
    ORDER BY 1
""").df()
yoy['growth_x'] = (yoy['orders_2018'] / yoy['orders_2017']).round(2)
print(yoy.to_string(index=False))

# 2018 monthly average vs the November 2017 peak
print()
print(conn.execute("""
    SELECT
        ROUND(AVG(orders))     AS avg_monthly_orders_2018,
        ROUND(AVG(revenue))    AS avg_monthly_revenue_2018
    FROM (
        SELECT strftime(o.order_purchase_timestamp, '%Y-%m') AS month,
               COUNT(DISTINCT o.order_id) AS orders,
               SUM(p.payment_value)       AS revenue
        FROM orders o
        JOIN payments p ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
          AND o.order_purchase_timestamp >= '2018-01-01'
          AND o.order_purchase_timestamp <  '2018-09-01'
        GROUP BY 1
    )
""").df().to_string(index=False))

month  orders_2017  orders_2018  growth_x
   01          750         7069      9.43
   02         1653         6555      3.97
   03         2546         7003      2.75
   04         2303         6798      2.95
   05         3546         6749      1.90
   06         3135         6099      1.95
   07         3872         6159      1.59
   08         4193         6351      1.51
   09         4150            0      0.00
   10         4478            0      0.00
   11         7289            0      0.00
   12         5513            0      0.00

 avg_monthly_orders_2018  avg_monthly_revenue_2018
                  6598.0                 1056622.0


![Monthly order and revenue trend](../reports/01_monthly_trend.png)

**Finding:**

The platform peaked in November 2017 at 7,289 orders / 1.15M BRL. In the first 8 months of 2018 it averaged 6,598 orders / 1.06M BRL per month — in other words, month-over-month growth stopped and the series settled onto a plateau.

Year over year, however, growth continues — it is only slowing: January 750 → 7,069 orders (9.4x), March 2,546 → 7,003 (2.75x), August 4,193 → 6,351 (1.51x). The growth multiple fell from 9.4x to 1.5x within 8 months.

**Business interpretation:**

These two readings lead to different decisions and must not be confused. The "sales have stalled" reading triggers an urgent demand-generation reflex; the correct reading is "the platform is moving from a hypergrowth phase to maturity". The 2018 plateau is not a collapse; it is the end of 2017's low base. Decision: the budget should be planned on the assumption that customer acquisition cost (CAC) can no longer be covered by the easy growth that came from a low base.

**Recommendation:**

Growth targets should be tracked through the YoY growth multiple rather than absolute order count, because the absolute series is misleading under seasonality and base effects. When setting targets for Q4 2018, November 2017 should be modeled not as a one-off peak but as a calendar effect that recurs every year (see Question 2).

**Limitations:**

- The series ends in August 2018; whether November 2018 repeated November 2017 cannot be verified with this data. The "growth is slowing" conclusion is therefore drawn without observing the busiest quarter of the year.
- The `delivered` filter slightly undercounts the most recent months (~2% in mid-2018), because orders delivered late are caught by the data cutoff. This biases the YoY multiple slightly downward against 2018; the slowdown is real, but the measured pace is a little below the true one.
- Revenue is in nominal BRL and is not adjusted for Brazilian inflation over 2016-2018. Real growth is somewhat lower than the figures here.
- Order count and revenue move together; the change in average basket size has not been separated out.
- The monthly chart is an aggregation and hides intra-month concentration: that the November 2017 peak is squeezed into a single day is only visible in the daily breakdown (see Question 2).

---

## Question 2: What drove the November 2017 peak?

In [3]:
# Busiest days of November 2017 and the Black Friday week's share of the month
# Note: no order_status filter here — this measures the full order flow on the campaign day.
print(conn.execute("""
    SELECT CAST(order_purchase_timestamp AS DATE) AS day, COUNT(*) AS orders
    FROM orders
    WHERE order_purchase_timestamp >= '2017-11-01' AND order_purchase_timestamp < '2017-12-01'
    GROUP BY 1 ORDER BY orders DESC LIMIT 5
""").df().to_string(index=False))

print()
print(conn.execute("""
    SELECT
        COUNT(*) AS november_total,
        SUM(CASE WHEN order_purchase_timestamp >= '2017-11-20'
                  AND order_purchase_timestamp <  '2017-11-28' THEN 1 ELSE 0 END) AS bf_week,
        ROUND(100.0 * SUM(CASE WHEN order_purchase_timestamp >= '2017-11-20'
                                AND order_purchase_timestamp <  '2017-11-28' THEN 1 ELSE 0 END)
              / COUNT(*), 1) AS bf_share_pct
    FROM orders
    WHERE order_purchase_timestamp >= '2017-11-01' AND order_purchase_timestamp < '2017-12-01'
""").df().to_string(index=False))

# "How many times" the peak — relative to what? The baseline is the MEDIAN of days outside the BF week.
# Using the mean would inflate the baseline with the peak itself (non-BF mean 187.9 vs median 174.5).
print()
print(conn.execute("""
    WITH daily AS (
        SELECT CAST(order_purchase_timestamp AS DATE) AS day, COUNT(*) AS orders
        FROM orders
        WHERE order_purchase_timestamp >= '2017-11-01' AND order_purchase_timestamp < '2017-12-01'
        GROUP BY 1
    )
    SELECT
        ROUND(MEDIAN(orders) FILTER (WHERE day < DATE '2017-11-20'
                                        OR day > DATE '2017-11-27'), 1)       AS non_bf_median,
        MAX(orders)                                                           AS peak_day,
        ROUND(MAX(orders) / MEDIAN(orders) FILTER (WHERE day < DATE '2017-11-20'
                                                      OR day > DATE '2017-11-27'), 1) AS peak_multiple
    FROM daily
""").df().to_string(index=False))

       day  orders
2017-11-24    1176
2017-11-25     499
2017-11-27     403
2017-11-26     391
2017-11-28     380

 november_total  bf_week  bf_share_pct
           7544   3411.0          45.2

 non_bf_median  peak_day  peak_multiple
         174.5      1176            6.7


![November 2017 daily order breakdown](../reports/04_nov2017_daily.png)

**Finding:**

The November 2017 peak is not month-long performance; it is an event squeezed into a single day. 24 November 2017 (Black Friday) alone produced 1,176 orders — against a median of 174.5 orders on the month's days outside the Black Friday week, i.e. **6.7x** normal daily volume. The 20-27 November window accounts for 3,411 of the month's 7,544 orders, or **45%**.

> The median, not the mean, was used as the baseline: days outside the BF week average 187.9 with a median of 174.5. Because the mean is affected by the pre-/post-campaign build-up, it shifts the definition of a "normal day" toward the peak and understates the effect.

**Business interpretation:**

This finding invalidates a recommendation of the type "let's adapt November's successful campaign strategies to other periods of the year". What created the peak is not an Olist-specific marketing tactic but Brazil's national shopping calendar. There is no way to repeat the same result in May; the demand itself is external. There is no "monthly strategy" to spread — there is a one-day demand spike.

The real decision area is not demand generation but **operational readiness for a predictable demand peak**: on 24 November, 6.7x the normal volume had to be handled with the same inventory and logistics capacity.

**Recommendation:**

Black Friday should be treated not as a marketing opportunity but as a capacity-planning event — because the demand is coming anyway; the risk is failing to serve it. Concretely: delivery time and satisfaction scores for orders sold during the November peak should be tracked separately against the annual average (see Question 4 — delivery time is the primary driver of satisfaction). The campaign budget should be shifted away from amplifying the peak toward the low-volume weeks before and after it.

The second, more fundamental question: do the customers won on this single day come back? The answer is in Question 5 — they don't. This directly limits Black Friday's value as a customer acquisition channel.

**Limitations:**

- There is only one Black Friday observation (2017). It cannot be separated whether the size of the effect is repeatable or was confounded with platform growth specific to 2017.
- The data contains no campaign/discount information. How much of the peak came from discount depth and how much from calendar awareness cannot be measured.
- The true incremental effect of the peak is unknown: some orders would have been placed without Black Friday and may simply have shifted in time. No check for a demand dip before/after November was performed.
- This breakdown has no `order_status` filter (it measures the full order flow on the campaign day), which is why the November total is 7,544 — this difference must be kept in mind when comparing with the `delivered`-based 7,289 in Question 1.

---

## Question 3: How is revenue distributed across categories?

In [4]:
# Category economics: revenue not on its own but together with units / price / freight load / satisfaction
print(conn.execute("""
    WITH item AS (
        SELECT COALESCE(t.product_category_name_english, p.product_category_name) AS category,
               oi.order_id, oi.price, oi.freight_value
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        LEFT JOIN read_csv_auto('../data/raw/product_category_name_translation.csv') t
               ON p.product_category_name = t.product_category_name
        WHERE p.product_category_name IS NOT NULL
    ),
    metrics AS (
        SELECT category,
               COUNT(*)                                       AS units_sold,
               ROUND(SUM(price))                              AS product_revenue,
               ROUND(AVG(price), 1)                           AS avg_price,
               ROUND(100.0*SUM(freight_value)/SUM(price), 1)  AS freight_load_pct
        FROM item GROUP BY 1
    ),
    -- Ratings are averaged at ORDER level: an item-level JOIN would repeat the same review for
    -- multi-item orders and distort the category average with its own weight.
    rating AS (
        SELECT u.category, ROUND(AVG(r.review_score), 2) AS avg_rating
        FROM (SELECT DISTINCT category, order_id FROM item) u
        JOIN reviews r ON u.order_id = r.order_id
        GROUP BY 1
    )
    SELECT m.*, r.avg_rating
    FROM metrics m LEFT JOIN rating r ON m.category = r.category
    ORDER BY m.product_revenue DESC
    LIMIT 10
""").df().to_string(index=False))

# Concentration
print()
print(conn.execute("""
    SELECT
        COUNT(*) AS category_count,
        ROUND(SUM(revenue)) AS total_product_revenue,
        ROUND(100.0 * SUM(CASE WHEN rank <=  5 THEN revenue ELSE 0 END) / SUM(revenue), 1) AS top5_share_pct,
        ROUND(100.0 * SUM(CASE WHEN rank <= 15 THEN revenue ELSE 0 END) / SUM(revenue), 1) AS top15_share_pct
    FROM (
        SELECT p.product_category_name, SUM(oi.price) AS revenue,
               ROW_NUMBER() OVER (ORDER BY SUM(oi.price) DESC) AS rank
        FROM products p JOIN order_items oi ON p.product_id = oi.product_id
        GROUP BY 1
    )
""").df().to_string(index=False))

# Lowest-satisfaction categories among the top 30 by revenue
print()
print(conn.execute("""
    WITH item AS (
        SELECT COALESCE(t.product_category_name_english, p.product_category_name) AS category,
               oi.order_id, oi.price
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        LEFT JOIN read_csv_auto('../data/raw/product_category_name_translation.csv') t
               ON p.product_category_name = t.product_category_name
        WHERE p.product_category_name IS NOT NULL
    ),
    top30 AS (SELECT category, SUM(price) AS revenue FROM item GROUP BY 1 ORDER BY revenue DESC LIMIT 30),
    rating AS (
        SELECT u.category, ROUND(AVG(r.review_score), 2) AS avg_rating
        FROM (SELECT DISTINCT category, order_id FROM item) u
        JOIN reviews r ON u.order_id = r.order_id GROUP BY 1
    )
    SELECT t.category, ROUND(t.revenue) AS product_revenue, r.avg_rating
    FROM top30 t JOIN rating r ON t.category = r.category
    ORDER BY r.avg_rating LIMIT 4
""").df().to_string(index=False))

             category  units_sold  product_revenue  avg_price  freight_load_pct  avg_rating
        health_beauty        9670        1258681.0      130.2              14.5        4.18
        watches_gifts        5991        1205006.0      201.1               8.3        4.07
       bed_bath_table       11115        1036989.0       93.3              19.7        3.97
       sports_leisure        8641         988049.0      114.3              17.1        4.17
computers_accessories        7827         911954.0      116.5              16.2        4.03
      furniture_decor        8334         729762.0       87.6              23.7        4.01
           cool_stuff        3796         635291.0      167.4              13.2        4.17
           housewares        6964         632249.0       90.8              23.1        4.14
                 auto        4235         592720.0      140.0              15.6        4.09
         garden_tools        4347         485256.0      111.6              20.4 

         category  product_revenue  avg_rating
 office_furniture         273961.0        3.62
home_construction          83088.0        3.97
   bed_bath_table        1036989.0        3.97
        telephony         323668.0        4.00


![Revenue by category — Top 15](../reports/02_category_revenue.png)

![Category economics: price × units × revenue × satisfaction](../reports/06_category_economics.png)

*The second chart shows why the first one is insufficient on its own: the revenue ranking (the bar chart above) reduces categories to a single dimension; the scatter below positions the same categories by price (x), units (y), revenue (bubble) and satisfaction (color) together.*

**Finding:**

74 categories produce 13.59M BRL in total product revenue (excluding freight). The distribution is concentrated: the top 5 categories account for **39.7%** of revenue and the top 15 for **76.3%**.

Revenue rank alone is misleading, however, because categories in the same revenue band have completely different economics:

| Category | Units | Avg. price | Revenue | Freight/revenue | Avg. rating |
|---|---|---|---|---|---|
| health_beauty | 9,670 | 130 BRL | 1.26M | 14.5% | 4.18 |
| watches_gifts | 5,991 | 201 BRL | 1.21M | 8.3% | 4.07 |
| bed_bath_table | 11,115 | 93 BRL | 1.04M | 19.7% | 3.97 |
| furniture_decor | 8,334 | 88 BRL | 0.73M | 23.7% | 4.01 |

`watches_gifts` produces most of its revenue from a high price (5,991 units × 201 BRL) and has the lowest freight load at 8.3% — this is a **price business**. `health_beauty` reaches almost the same revenue with close to twice the units (9,670 × 130 BRL) — this is a **volume business**. The two sit in the same revenue band but run on entirely different levers.

The weakest economics belong to `bed_bath_table`: despite being the best-selling category on the list (11,115 units), it generates less revenue at a 93 BRL average price, its freight load is more than twice as high (19.7%) and its satisfaction is in the lowest group (3.97).

Among the top 30 categories by revenue, the lowest satisfaction belongs to `office_furniture` (3.62). Despite its revenue contribution, this category is not a growth candidate but an operational problem to be investigated first.

**Business interpretation:**

If category investment decisions are made by revenue rank, resources go to the wrong place. High-volume / low-price / heavy-freight categories generate more operational load and more satisfaction risk with every additional order. The same 1M BRL of revenue costs noticeably less when it comes through `watches_gifts` than when it comes through `bed_bath_table`.

The lever also depends on the category type: in volume categories (`bed_bath_table`, `furniture_decor`) the gain lies in freight cost, in price categories (`watches_gifts`) it lies in conversion rate. The same campaign budget does not do the same job in both groups.

**Recommendation:**

Seller acquisition and advertising budget should be directed primarily to `health_beauty` and `watches_gifts` — because these two categories are both revenue leaders and have a low freight load and above-average satisfaction scores, so growing them does not create operational cost or rating risk. For `bed_bath_table` and `furniture_decor`, freight cost and delivery performance should be addressed before growing volume; growing them in their current state also grows the number of low-rated orders. `office_furniture` should go through a root-cause analysis before any budget decision.

**Limitations:**

- **No margin data.** The entire analysis is based on revenue; which category is actually profitable is unknown. Freight load is a proxy indicator, not true unit economics. The ranking could differ on a margin basis.
- The average rating is computed **at order level** (`DISTINCT category, order_id`). With an item-level JOIN, the review of a multi-item order would be counted over and over; this method removes that error, which was pulling the ratings in the table down by ~0.03-0.10. Earlier item-based values in the report (e.g. `bed_bath_table` 3.90) have been updated with this correction.
- The rating is still given to the whole order, not to a single item. In a multi-category order, a bad rating is written to every category — the category-rating relationship is therefore still somewhat blurred and should be used for ranking categories against each other rather than for absolute values.
- This table excludes rows where `product_category_name IS NULL`: 610 products, 1,603 order items and 179,535 BRL of revenue (1.3% of the total). The concentration query (74 categories / 13.59M BRL), on the other hand, counts this group as a separate category — which is why the totals of the two queries don't match exactly.
- Demand elasticity by category was not measured: the recommendation "shift the budget here" assumes that additional budget will turn into sales in these categories.

---

## Question 4: How does delivery time affect customer satisfaction?

In [5]:
# Delivery time distribution: should the mean or the median be reported?
print(conn.execute("""
    SELECT
        ROUND(MEDIAN(days), 1)               AS median_days,
        ROUND(AVG(days), 1)                  AS mean_days,
        ROUND(QUANTILE_CONT(days, 0.90), 1)  AS p90_days,
        MAX(days)                            AS max_days
    FROM (
        SELECT DATE_DIFF('day', o.order_purchase_timestamp, o.order_delivered_customer_date) AS days
        FROM orders o
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    )
""").df().to_string(index=False))

# Satisfaction by delivery time band
print()
print(conn.execute("""
    SELECT
        CASE WHEN days <=  3 THEN '0-3 days'
             WHEN days <=  7 THEN '4-7 days'
             WHEN days <= 14 THEN '8-14 days'
             WHEN days <= 30 THEN '15-30 days'
             ELSE '30+ days' END                                       AS delivery_band,
        COUNT(*)                                                       AS orders,
        ROUND(AVG(rating), 2)                                          AS avg_rating,
        ROUND(100.0 * SUM(CASE WHEN rating <= 2 THEN 1 ELSE 0 END) / COUNT(*), 1) AS low_rating_pct
    FROM (
        SELECT DATE_DIFF('day', o.order_purchase_timestamp, o.order_delivered_customer_date) AS days,
               r.review_score AS rating
        FROM orders o
        JOIN reviews r ON o.order_id = r.order_id
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    )
    GROUP BY 1
    ORDER BY MIN(days)
""").df().to_string(index=False))

# --- Adherence to the promised date ---
# DEFINITION NOTE: the time component of order_estimated_delivery_date is 00:00:00 in every row;
# in other words, the dataset carries a DAY commitment, not a TIME. A raw timestamp comparison
# (delivered > estimated) therefore counts an order delivered within the promised day, e.g. at
# 14:00, as "late". The comparison must be made at day level.
# The first query below shows both the size of this distinction and why it matters:
# the misclassified group behaves like on-time deliveries, not like late ones.
print()
print(conn.execute("""
    SELECT CASE
             WHEN CAST(o.order_delivered_customer_date AS DATE)
                > CAST(o.order_estimated_delivery_date AS DATE)   THEN '1. Actually late (day overrun)'
             WHEN o.order_delivered_customer_date
                > o.order_estimated_delivery_date                 THEN '2. Promised day, later hour'
             ELSE                                                      '3. On time'
           END                                AS status,
           COUNT(*)                           AS orders,
           ROUND(AVG(r.review_score), 2)      AS avg_rating
    FROM orders o
    JOIN reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df().to_string(index=False))

# The two groups under the corrected definition (these are the reported numbers)
print()
print(conn.execute("""
    SELECT
        CASE WHEN CAST(o.order_delivered_customer_date AS DATE)
                > CAST(o.order_estimated_delivery_date AS DATE)
             THEN 'LATE' ELSE 'ON TIME' END                                     AS status,
        COUNT(*)                                                                AS orders,
        ROUND(AVG(r.review_score), 2)                                           AS avg_rating,
        ROUND(100.0 * SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END)
              / COUNT(*), 1)                                                    AS low_rating_pct
    FROM orders o
    JOIN reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY 1
""").df().to_string(index=False))

# Rates across all delivered orders, without the review filter
print()
print(conn.execute("""
    SELECT
        COUNT(*)                                                                AS delivered,
        SUM(CASE WHEN CAST(order_delivered_customer_date AS DATE)
                    > CAST(order_estimated_delivery_date AS DATE)
                 THEN 1 ELSE 0 END)                                             AS late,
        ROUND(100.0 * SUM(CASE WHEN CAST(order_delivered_customer_date AS DATE)
                                  > CAST(order_estimated_delivery_date AS DATE)
                               THEN 1 ELSE 0 END) / COUNT(*), 1)                AS late_pct,
        SUM(CASE WHEN DATE_DIFF('day', order_purchase_timestamp,
                                order_delivered_customer_date) > 30
                 THEN 1 ELSE 0 END)                                             AS over_30_days,
        ROUND(100.0 * SUM(CASE WHEN DATE_DIFF('day', order_purchase_timestamp,
                                              order_delivered_customer_date) > 30
                               THEN 1 ELSE 0 END) / COUNT(*), 1)                AS over_30_pct
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
""").df().to_string(index=False))

 median_days  mean_days  p90_days  max_days
        10.0       12.5      23.0       210



delivery_band  orders  avg_rating  low_rating_pct
     0-3 days    6951        4.46             7.0
     4-7 days   23728        4.40             7.7
    8-14 days   37985        4.30             9.1
   15-30 days   23484        3.94            16.4
     30+ days    4205        2.20            64.5

                        status  orders  avg_rating
1. Actually late (day overrun)    6409        2.27
   2. Promised day, later hour    1291        4.03
                    3. On time   88653        4.29

 status  orders  avg_rating  low_rating_pct
ON TIME   89944        4.29             9.3
   LATE    6409        2.27            62.4

 delivered   late  late_pct  over_30_days  over_30_pct
     96470 6534.0       6.8        4294.0          4.5


In [6]:
# How much does the size of the overrun affect the rating? (Is "late" binary, or graduated?)
# This breakdown directly sizes the recommendation: if most delays are small overruns,
# adding a buffer to the estimated date moves them to the on-time side without touching delivery speed.
print(conn.execute("""
    SELECT
        CASE WHEN d <=  0 THEN 'On time'
             WHEN d  =  1 THEN '1 day'
             WHEN d <=  3 THEN '2-3 days'
             WHEN d <=  7 THEN '4-7 days'
             WHEN d <= 15 THEN '8-15 days'
             ELSE '15+ days' END           AS promise_overrun,
        COUNT(*)                           AS orders,
        ROUND(AVG(r.review_score), 2)      AS avg_rating
    FROM (
        SELECT o.order_id,
               DATE_DIFF('day', CAST(o.order_estimated_delivery_date AS DATE),
                                CAST(o.order_delivered_customer_date AS DATE)) AS d
        FROM orders o
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    ) x
    JOIN reviews r ON x.order_id = r.order_id
    GROUP BY 1
    ORDER BY MIN(d)
""").df().to_string(index=False))

promise_overrun  orders  avg_rating
        On time   89944        4.29
          1 day     823        3.73
       2-3 days    1033        2.94
       4-7 days    1756        2.10
      8-15 days    1609        1.68
       15+ days    1188        1.73


![Delivery time distribution and its relationship with satisfaction](../reports/05_delivery_satisfaction.png)

**Finding:**

The delivery time distribution is **right-skewed**: median 10 days, mean 12.5 days, 90th percentile 23 days. The gap shows why an "average delivery time" metric on its own is misleading — what pulls the mean up is not the typical customer's experience but the long tail.

There is a strong, one-directional relationship between delivery time and satisfaction:

| Delivery time | Orders | Avg. rating | 1-2 star share |
|---|---|---|---|
| 0-3 days | 6,951 | 4.46 | 7.0% |
| 4-7 days | 23,728 | 4.40 | 7.7% |
| 8-14 days | 37,985 | 4.30 | 9.1% |
| 15-30 days | 23,484 | 3.94 | 16.4% |
| **30+ days** | **4,205** | **2.20** | **64.5%** |

The effect sharpens after day 15 and collapses past 30 days: **64.5% of customers who waited 30+ days give 1-2 stars**. This band is only 4.4% of total volume but produces a disproportionately large share of bad ratings.

Adherence to the promised date is even more decisive: the 6,409 orders that overran the estimated delivery day (6.7% of reviewed deliveries) have an average rating of **2.27**, with **62.4% at 1-2 stars**; the 89,944 orders delivered on time average **4.29**, with a bad-rating share of **9.3%**. The **2.02-point gap** between them is of a size that no other variable in the dataset produces.

> **Definition note — how "late" was measured:** the time component of the `order_estimated_delivery_date` field is `00:00:00` in every row; in other words, the dataset carries a **day** commitment, not a time. A raw timestamp comparison (`delivered > estimated`) therefore also counts orders delivered within the promised day as late. The 1,291 orders misclassified this way behave like on-time deliveries, not late ones: their average rating is **4.03**, versus 2.27 for the genuinely late group. The comparison is therefore made at day level (`CAST(... AS DATE)`). Measured with the raw definition, the late group appears as 7,700 orders / 2.57 rating and the gap drops from 2.02 to 1.72 — in other words, the faulty definition was making the finding look **weaker** than it is.

> Note: the numbers above cover orders with a review. Across all delivered orders (96,470 orders), late deliveries number 6,534 (6.8%) and deliveries taking 30+ days number 4,294 (4.5%).

**Business interpretation:**

The primary driver of satisfaction is not product or price but logistics — and this is an area where the platform can intervene directly. Moreover, what matters is not absolute duration but **keeping the promise**: a delivery that takes 30 days but was promised in 35 gets a better rating than one that takes 12 days but was promised in 8. This means the estimated-delivery-date calculation is itself a satisfaction lever.

The size of the overrun also matters, and the rating loss is **graduated**, not binary:

| Promise overrun | Orders | Avg. rating |
|---|---|---|
| On time | 89,944 | 4.29 |
| 1 day | 823 | 3.73 |
| 2-3 days | 1,033 | 2.94 |
| 4-7 days | 1,756 | 2.10 |
| 8-15 days | 1,609 | 1.68 |
| 15+ days | 1,188 | 1.73 |

A one-day delay lowers the rating by 0.56, 4-7 days by 2.19, and after 8 days it saturates at ~1.7. This gradation is a direct lever: **29% of delays are overruns of only 1-3 days** (1,856 orders), and these can be moved to the "on time" side by adding a buffer of a few days to the estimated date — without touching delivery speed at all.

The skew of the distribution also has a direct management consequence: if the target is set as "reduce average delivery time to X days", effort goes into the middle of the distribution, while the rating loss is produced in the tail. The right target is not to improve the average but to **cut the tail**.

**Recommendation:**

Two parallel actions should be taken:

1. **The 4,294 orders that took 30+ days** (4.5% of all deliveries) should be examined by seller and route, and a delivery SLA should be defined for sellers where the problem concentrates — because this 4.5% slice produces a disproportionately large share of bad ratings.
2. **The estimated delivery date should be calculated more conservatively.** The late-order rate is 6.8% and their rating loss is 2.02; moreover, 29% of these delays are small 1-3 day overruns. Adding a buffer of a few days to the estimated date eliminates a significant part of this loss without changing delivery speed at all. This is the cheapest win, requiring no logistics investment.

The tracked metric should change accordingly: instead of average delivery time, **p90 delivery time (23 days)** and **promise-adherence rate (93.2%)** should go on the dashboard.

**Limitations:**

- This is a **correlation, not causation**. There may be a common third variable behind long delivery and bad ratings: remote regions, problematic sellers or out-of-stock products may be both delivered late and rated poorly. Seller and geography were not controlled for.
- There is review bias: customers with a bad experience are more inclined to leave a review. The rating distribution may not represent true satisfaction one-to-one.
- The analysis covers only **delivered** orders. Canceled orders or orders that never arrived — probably the worst experiences — are completely excluded.
- The recommendation to "push the estimated date out" assumes that a longer delivery promise does not reduce the purchase decision. This conversion effect cannot be measured with this data and requires an A/B test. The calculation that adding a buffer turns delays into "on time" is also mechanical: if the customer's expectation threshold shifts with the buffer, the gain will be smaller.
- In the histogram, deliveries over 60 days are collapsed into the last bin for readability; the right end of the chart does not show the true maximum (210 days), but the 30+ band in the table includes these orders in full.

---

## Question 5: Do customers buy again?

In [7]:
# --- Repeat customer rate ---
# DEFINITION NOTE: in Olist, a multi-seller basket is recorded as a separate order_id per seller.
# The "order count > 1" criterion therefore counts a single basket, created seconds apart on the
# same day, as a repeat purchase. A shopping event should be counted as a PURCHASE DAY, not an
# order. The first query below puts the two definitions side by side.
print("Raw definition (order-based) — the faulty measurement in the original report:")
print(conn.execute("""
    SELECT
        CASE WHEN order_count > 1 THEN 'Repeat' ELSE 'One-time' END AS segment,
        COUNT(*)                                                     AS customers,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)           AS customer_share_pct,
        ROUND(SUM(revenue))                                          AS revenue,
        ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 1)   AS revenue_share_pct
    FROM (
        SELECT c.customer_unique_id,
               COUNT(DISTINCT o.order_id) AS order_count,
               SUM(p.payment_value)       AS revenue
        FROM customers c
        JOIN orders   o ON c.customer_id = o.customer_id
        JOIN payments p ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    GROUP BY 1
""").df().to_string(index=False))

print()
print("Corrected definition (purchase-day based) — these are the reported numbers:")
print(conn.execute("""
    SELECT
        CASE WHEN purchase_days > 1 THEN 'Repeat' ELSE 'One-time' END AS segment,
        COUNT(*)                                                     AS customers,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)           AS customer_share_pct,
        ROUND(SUM(revenue))                                          AS revenue,
        ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 1)   AS revenue_share_pct
    FROM (
        SELECT c.customer_unique_id,
               COUNT(DISTINCT CAST(o.order_purchase_timestamp AS DATE)) AS purchase_days,
               SUM(p.payment_value)                                     AS revenue
        FROM customers c
        JOIN orders   o ON c.customer_id = o.customer_id
        JOIN payments p ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    GROUP BY 1
""").df().to_string(index=False))

# Source of the gap between the two definitions: how many "repeat" customers are really a split same-day basket?
print()
print(conn.execute("""
    SELECT COUNT(*)                                                          AS raw_repeat,
           SUM(CASE WHEN day_count = 1 THEN 1 ELSE 0 END)                    AS all_same_day,
           ROUND(100.0 * SUM(CASE WHEN day_count = 1 THEN 1 ELSE 0 END)
                 / COUNT(*), 1)                                              AS same_day_pct
    FROM (
        SELECT c.customer_unique_id,
               COUNT(DISTINCT o.order_id)                                AS orders,
               COUNT(DISTINCT CAST(o.order_purchase_timestamp AS DATE))  AS day_count
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    WHERE orders > 1
""").df().to_string(index=False))

Raw definition (order-based) — the faulty measurement in the original report:


 segment  customers  customer_share_pct    revenue  revenue_share_pct
  Repeat       2801                 3.0   864357.0                5.6
One-time      90556                97.0 14558105.0               94.4

Corrected definition (purchase-day based) — these are the reported numbers:


 segment  customers  customer_share_pct    revenue  revenue_share_pct
  Repeat       2015                2.16   646454.0                4.2
One-time      91342               97.84 14776008.0               95.8

 raw_repeat  all_same_day  same_day_pct
       2801         786.0          28.1


In [8]:
# --- Right-censoring check: is the rate truly low, or is the observation window too short? ---
# A customer who placed their first order in August 2018 had no time to place a second. Two
# censoring-free measurements: (1) "ever repeated" in the mature cohort whose first purchase was
# 12+ months ago, (2) a fixed 90-day window of equal length for everyone. Reference cutoff:
# 29 August 2018 (the last complete month of the series; Sep-Oct 2018 hold 20 orders in total,
# a data-cutoff remnant).
print(conn.execute("""
    WITH days AS (
        SELECT c.customer_unique_id AS uid,
               CAST(o.order_purchase_timestamp AS DATE) AS day
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1, 2
    ),
    customer AS (
        SELECT uid, MIN(day) AS first_day, COUNT(*) AS purchase_days
        FROM days GROUP BY 1
    )
    SELECT
        COUNT(*)                                                       AS mature_customers,
        SUM(CASE WHEN purchase_days > 1 THEN 1 ELSE 0 END)             AS repeat_customers,
        ROUND(100.0 * SUM(CASE WHEN purchase_days > 1 THEN 1 ELSE 0 END)
              / COUNT(*), 2)                                           AS repeat_pct
    FROM customer
    WHERE first_day <= DATE '2018-08-29' - INTERVAL 12 MONTH
""").df().to_string(index=False))

print()
print(conn.execute("""
    WITH days AS (
        SELECT c.customer_unique_id AS uid,
               CAST(o.order_purchase_timestamp AS DATE) AS day
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1, 2
    ),
    first_purchase AS (SELECT uid, MIN(day) AS first_day FROM days GROUP BY 1)
    SELECT
        COUNT(*)                                                       AS observable_customers,
        SUM(CASE WHEN second_day IS NOT NULL THEN 1 ELSE 0 END)            AS d90_repeat,
        ROUND(100.0 * SUM(CASE WHEN second_day IS NOT NULL THEN 1 ELSE 0 END)
              / COUNT(*), 2)                                           AS d90_pct
    FROM (
        SELECT f.uid,
               MIN(CASE WHEN d.day > f.first_day
                         AND d.day <= f.first_day + 90 THEN d.day END) AS second_day
        FROM first_purchase f
        JOIN days d ON f.uid = d.uid
        WHERE f.first_day <= DATE '2018-08-29' - INTERVAL 90 DAY
        GROUP BY 1
    )
""").df().to_string(index=False))

 mature_customers  repeat_customers  repeat_pct
            21361             795.0        3.72

 observable_customers  d90_repeat  d90_pct
                75387       978.0      1.3


**Finding:**

Repeat purchasing is practically absent. Of the 93,357 unique customers who placed a delivered order, only **2,015 (2.16%)** shopped more than once; this group accounts for **4.2%** of total revenue. In other words, **95.8% of revenue comes from one-time customers**.

> **Definition note — how "repeat" was counted:** in Olist, a multi-seller basket is recorded as a separate `order_id` per seller. The "order count > 1" criterion therefore counts **a single basket** created seconds apart on the same day as a repeat purchase. Under the raw definition, 2,801 customers (3.0%) look like repeat customers, but **786 of them (28.1%)** placed all their orders on the same day — there was never a second purchase decision. A shopping event is therefore counted as a **purchase day**, not an order. The same correction holds when all order statuses are included: across 96,096 customers, the raw rate is 3.12% and the clean rate **2.24%**.

The right-censoring check does not change the result; it only sharpens it. A short observation window could be pulling the rate down (a customer who placed their first order in August 2018 had no time to place a second), but the censoring-free measurements give the same picture:

| Measurement | Base | Repeat | Rate |
|---|---|---|---|
| Raw (the faulty definition in the original report) | 93,357 customers | 2,801 | 3.00% |
| Corrected (purchase day) | 93,357 customers | 2,015 | **2.16%** |
| Mature cohort (first purchase 12+ months ago, "ever repeated") | 21,361 customers | 795 | **3.72%** |
| Fixed 90-day window (second purchase) | 75,387 customers | 978 | **1.30%** |

So even when the observation window is fully open — for customers tracked for more than a year — the repeat rate does not exceed **3.7%**. In a fixed 90-day window, only **1.3%** of customers make a second purchase. Censoring does not explain this finding.

**Business interpretation:**

This finding invalidates, from the outset, every recommendation of the type "let's grow revenue from existing customers" — there is no base to play on. The platform runs entirely on a new-customer acquisition engine, and this may not be a malfunction but a structural consequence of the business model: Olist is a marketplace, and the customer relationship and brand loyalty largely stay with the seller.

This also explains the mechanism behind the growth slowdown in Question 1: because no base revenue accumulates from repeat purchases, each month's revenue has to be regenerated from the new customers acquired that month. When growth stops, there is no cushion against decline.

**Recommendation:**

Before allocating budget to retention campaigns, it should be determined whether this 2-4% band is structural or fixable — because the two require entirely different investments and the current data cannot make that distinction. Concrete first step: the delivery time and rating of the 2,015 repeat customers' first purchase should be compared with those of one-time customers. If repeat customers had a markedly better first experience, retention is an **operations** problem rather than a marketing one, and is covered by the same investment as the actions in Question 4.

Until this is resolved, budget priority should stay on customer acquisition and the first-order experience.

**Limitations:**

- The mature cohort measurement (3.72%) covers only 21,361 customers, and they are the platform's **early** customers. Early users typically behave differently; this rate cannot be directly generalized to 2018 customers.
- The accuracy of the `customer_unique_id` mapping is relied upon. If the same person re-registered with a different email/address, they are counted as two separate customers and the repeat rate looks lower than it is. This is the only source of bias the report cannot measure that could push the rate up.
- The "purchase day" definition is not perfect either: a customer who genuinely made two separate purchase decisions on the same day is counted once. This pulls the rate down somewhat — but it is much smaller than the split-basket error in the opposite direction.
- This is marketplace data; a customer may have bought again from the same seller outside the platform, which is invisible in this data.
- There is no industry benchmark. Calling 2-4% a "problem" is premature without knowing whether it is low or normal for Brazilian e-commerce marketplaces.

---

## General limitations

Constraints that apply to this report as a whole:

1. **Time window:** October 2016 – August 2018, 22 months. One complete Black Friday, one complete year-over-year comparison. Seasonality cannot be modeled reliably.
2. **No margin or cost data.** All "performance" assessments are based on revenue. The profitability ranking could differ from the one here.
3. **No causal claims.** No finding is experimental; all are observational correlations. In particular, the delivery-satisfaction relationship in Question 4 should be tested in a controlled way before being turned into action.
4. **No uncertainty measures reported.** The averages in the tables are point estimates; no confidence intervals or significance tests are given. For large groups (n > 5,000) this is not a problem, but differences in small cells (e.g. 1-day delay, n = 823) need care when read.
5. **No inflation adjustment.** BRL amounts are nominal.
6. **Right censoring:** delivery, reviews and repeat purchases for orders placed in the final months are caught by the data cutoff. The last 1-2 months should not be used in any trend interpretation. In Question 5 this effect was measured separately and shown not to change the result.
7. **Aggregation and definition risk:** every monthly/category-level metric hides the distribution beneath it. In this report, the analysis deliberately went one level deeper in two places (November → daily, revenue → units/price/freight/rating), and in both the top-level interpretation changed. In addition, the **definitions** of two metrics were wrong in the first measurement (late delivery, repeat purchase) and their magnitudes changed once corrected. The same suspicion applies to the other, unaudited metrics.

---

## Chart index

Charts are generated by `03_eda_charts.ipynb` and saved under `reports/`.

| File | Where used | What it shows |
|---|---|---|
| `01_monthly_trend.png` | Question 1 | Monthly order and revenue series (`COUNT(DISTINCT order_id)`, Sep-Oct 2018 data cutoff excluded) |
| `02_category_revenue.png` | Question 3 | Top 15 categories by revenue |
| `04_nov2017_daily.png` | Question 2 | November 2017 daily breakdown, Black Friday week highlighted |
| `05_delivery_satisfaction.png` | Question 4 | Delivery time distribution + volume and 1-2 star share by band |
| `06_category_economics.png` | Question 3 | Category economics: price × units, bubble = revenue, color = rating |

`03_rfm_segmentation.png` was produced during the exploration phase and is not included in this report — rationale below (analyses not covered, item 4).

---

## Analyses not covered by this report

Candidate next steps, ranked by impact potential:

| # | Analysis | Why it matters |
|---|---|---|
| 1 | **Seller concentration (Pareto)** | How many sellers revenue depends on is the single most critical risk metric in a marketplace, and it was not measured in this report at all. The same analysis also shows whether poor delivery performance clusters in particular sellers |
| 2 | **Geography × seller controlled delivery analysis** | The biggest gap in Question 4: how much of the relationship really comes from delivery time, and how much from common variables such as a remote state or a problematic seller. A comparison stratified within state/seller largely resolves this |
| 3 | **Payment / installment analysis** | The relationship between installment count and basket size; directly produces pricing decisions in the Brazilian market |
| 4 | **RFM segmentation** | Step 5.3 in the guide. Note: since frequency = 1 for 98% of customers, the F score is almost constant and the segments collapse into R — this constraint is itself a more valuable finding than the RFM table |
| 5 | **Basket size decomposition** | How much of the revenue change comes from order count and how much from average basket (the missing half of Question 1) |
| 6 | **First experience of repeat customers** | The question Question 5 leaves open: did the 2,015 repeat customers get better delivery and ratings on their first purchase? Tells whether retention is a marketing or an operations problem |
| 7 | **Return of the Black Friday cohort** | The repeat rate of customers acquired on 24 November should be compared with those acquired on other days; combines Question 2 and Question 5 to measure the campaign's acquisition value |